# Fase 4.3 — Interpretabilidad: coeficientes, *prior* frente a posterior, SHAP y supervivencia

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Un modelo de riesgo que no se puede explicar no se puede defender: ni ante un comité de crédito, ni ante la SFC, ni ante un evaluador académico. En este notebook ajusto los modelos principales con **todos** los 733 créditos, con sus hiperparámetros elegidos por el mismo pliegue interno de la validación, e interpreto lo que aprendieron.

- **M2 (modelo elegido):** coeficientes estandarizados, *odds ratios* por desviación estándar e IC por *bootstrap* de clientes.
- **M3 (*prior* de Home Credit):** posterior frente a *prior*, con intervalos creíbles de Laplace verificados con un muestreo de Metropolis.
- **M5 (*boosting* monótono):** valores SHAP y perfiles de dependencia.
- **M6 y Cox:** *hazard ratios* del tiempo hasta 90+, que aprovechan los créditos censurados.
- **Contraste de signos** de todos los modelos frente a la hipótesis de dominio.

Advertencia de lectura: con 23 eventos, los coeficientes son **señales, no efectos estimados con precisión**. Los IC lo dejan a la vista.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import statsmodels.api as sm
from joblib import Parallel, delayed
from scipy.special import expit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from tesis_v2 import config, contexto, diseno, figuras, io_datos, metricas as mt, modelos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180, "display.max_columns", 30)
figuras.estilo()

D, ctx, ds = contexto.cargar()
X = diseno.construir_X(D)
V = diseno.NUCLEO
SIG = pd.Series(diseno.SIGNO_ESPERADO)
print(f"{len(D)} créditos, {int(D['y'].sum())} eventos")

.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


733 créditos, 23 eventos


## 1. M2, logística penalizada (modelo elegido)

El pliegue interno sobre los 733 créditos elige la penalización y C. Los IC salen de 1.000 réplicas *bootstrap* por cliente, reajustando con esa misma configuración. Son IC del **estimador penalizado**: están centrados en coeficientes encogidos hacia 0, que es justamente lo que lo hace estable.

In [2]:
m2 = modelos.M2Penalizada().ajustar(D, ctx)
C, l1 = m2.elegido
print(f"Configuración elegida: {'L2' if l1 == 0 else ('L1' if l1 == 1 else f'EN {l1}')}, C = {C:.3g}")
coef2 = m2.coeficientes(D)


def coef_boot(idx):
    Db = D.iloc[idx].reset_index(drop=True)
    return modelos._Logistica((C, l1)).ajustar(Db, ctx).pipe[-1].coef_[0]


idxs = list(mt.indices_bootstrap(D["grupo_cliente"], B=1000, semilla=config.SEMILLA + 3))
B2 = np.array(Parallel(n_jobs=10)(delayed(coef_boot)(i) for i in idxs))
t2 = pd.DataFrame({"variable": V, "coef_estandarizado": coef2.to_numpy(), "ic_inf": np.percentile(B2, 2.5, axis=0),
                   "ic_sup": np.percentile(B2, 97.5, axis=0), "odds_ratio_por_de": np.exp(coef2.to_numpy()),
                   "signo_esperado": SIG[V].map({1: "+", -1: "−", 0: "?"}).to_numpy(),
                   # Para el monto (sin signo esperado) es la probabilidad de un coeficiente positivo.
                   "prob_signo_esperado": [np.mean(np.sign(B2[:, j]) == (SIG[v] if SIG[v] != 0 else 1))
                                           for j, v in enumerate(V)]})
io_datos.guardar_resultado(t2.round(4), "fase4", "coeficientes_M2")
t2.round(3)

Configuración elegida: L2, C = 0.1


,variable,coef_estandarizado,ic_inf,ic_sup,odds_ratio_por_de,signo_esperado,prob_signo_esperado
0,log_monto,0.239,-0.018,0.482,1.269,?,0.965
1,log_plazo,0.048,-0.257,0.311,1.049,+,0.611
2,edad_t0,-0.199,-0.480,0.049,0.820,−,0.931
3,tiene_historia,0.031,-0.253,0.263,1.032,−,0.437
4,log_n_previos,-0.016,-0.234,0.231,0.984,−,0.590
5,meses_cliente,-0.168,-0.377,0.029,0.846,−,0.949
6,log_peor_mora_previa,-0.031,-0.355,0.264,0.969,+,0.416
7,previo_adverso,0.045,-0.250,0.315,1.046,+,0.611
8,previo_favorable,-0.071,-0.288,0.144,0.932,−,0.734
9,n_activos_t0,-0.134,-0.394,0.127,0.875,+,0.174


**Lectura de M2.** Ningún coeficiente tiene un IC 95 % que excluya estrictamente el 0. Es la consecuencia directa de tener 23 eventos. La probabilidad *bootstrap* de signo permite, aun así, separar señales consistentes de ruido:

- **Señales consistentes:**
  - la **experiencia del empleador** (OR 0,72 por desviación estándar; signo esperado en el 97 % de las réplicas; límite superior del IC en 0,005);
  - la **antigüedad como cliente** (OR 0,85; 95 %);
  - la **edad** (OR 0,82; 93 %);
  - el **monto**, que aumenta el riesgo (OR 1,27; límite inferior del IC en −0,02).

  Coinciden con lo que la Fase 2 anticipaba con el WoE.
- **Resto de la historia** (tiene historia, número de previos, tipo de cierre): con la penalización queda cerca de 0.
- **Signos contrarios a la hipótesis:** `log_peor_mora_previa` (ligeramente negativo) y `n_activos_t0` (negativo), con IC que incluyen el 0. No los interpreto como efectos: con 23 eventos, la probabilidad de signo correcto de esas variables está cerca del 50 %.
- **Cautela con el empleador.** Su efecto mezcla dos cosas: la madurez de la relación con el convenio y el calendario (crece con el tiempo y el riesgo de la cartera bajó con las cohortes). La sensibilidad sin la cohorte 2022 del notebook 12 ayuda a separarlas.

## 2. M3: qué aporta el *prior* de Home Credit

En M3 cada coeficiente parte del centro que sugiere Home Credit (o de 0 si la variable no existe allí) y los datos de FINTECH lo mueven. Con τ = 0,25 elegido por el pliegue interno, el *prior* es fuerte: una desviación estándar de 0,25 sobre la escala estandarizada. Muestro el centro del *prior*, la media posterior y su intervalo creíble del 95 %.

El intervalo de Laplace supone que la posterior es normal. Lo verifico con un muestreo de Metropolis (40.000 iteraciones, propuesta normal con la covarianza de Laplace escalada).

In [3]:
m3 = modelos.M3PriorHomeCredit().ajustar(D, ctx)
f3 = m3.m
print(f"τ elegido: {m3.elegido}")
Xs = f3.prep.transform(X)
yv = D["y"].to_numpy(float)
th_map = np.r_[f3.b0, f3.b]


def log_post(th):
    z = th[0] + Xs @ th[1:]
    return np.sum(yv * z - np.logaddexp(0, z)) - np.sum((th[1:] - f3.mu) ** 2) / (2 * f3.tau ** 2)


rng = np.random.default_rng(config.SEMILLA)
L = np.linalg.cholesky(f3.cov) * 2.38 / np.sqrt(len(th_map))
th, lp, cadena, acept = th_map.copy(), log_post(th_map), [], 0
for it in range(40_000):
    prop = th + L @ rng.standard_normal(len(th))
    lpp = log_post(prop)
    if np.log(rng.random()) < lpp - lp:
        th, lp, acept = prop, lpp, acept + 1
    if it >= 5_000 and it % 5 == 0:
        cadena.append(th.copy())
cadena = np.array(cadena)
print(f"Tasa de aceptación de Metropolis: {acept / 40_000:.2f}")
de = np.sqrt(np.diag(f3.cov))[1:]
t3 = pd.DataFrame({"variable": V, "centro_prior": f3.mu, "media_posterior_MAP": f3.b,
                   "ic_laplace": [f"{a:.3f} a {b:.3f}" for a, b in zip(f3.b - 1.96 * de, f3.b + 1.96 * de)],
                   "ic_metropolis": [f"{a:.3f} a {b:.3f}" for a, b in zip(*np.percentile(cadena[:, 1:], [2.5, 97.5], axis=0))],
                   "desplazamiento_por_datos": f3.b - f3.mu, "coef_M2": coef2.to_numpy()})
io_datos.guardar_resultado(t3.round(4), "fase4", "posterior_M3")
t3.round(3)

τ elegido: 0.25


Tasa de aceptación de Metropolis: 0.26


,variable,centro_prior,media_posterior_MAP,ic_laplace,ic_metropolis,desplazamiento_por_datos,coef_M2
0,log_monto,0.000,0.210,-0.131 a 0.550,-0.131 a 0.555,0.210,0.239
1,log_plazo,-0.088,0.008,-0.327 a 0.343,-0.303 a 0.353,0.096,0.048
2,edad_t0,-0.231,-0.270,-0.605 a 0.065,-0.624 a 0.031,-0.038,-0.199
3,tiene_historia,0.352,0.241,-0.161 a 0.643,-0.178 a 0.640,-0.111,0.031
4,log_n_previos,-0.082,-0.161,-0.591 a 0.268,-0.585 a 0.263,-0.079,-0.016
5,meses_cliente,-0.097,-0.242,-0.664 a 0.179,-0.679 a 0.162,-0.146,-0.168
6,log_peor_mora_previa,0.263,0.105,-0.270 a 0.481,-0.293 a 0.481,-0.158,-0.031
7,previo_adverso,0.000,0.013,-0.354 a 0.379,-0.359 a 0.368,0.013,0.045
8,previo_favorable,0.000,-0.092,-0.468 a 0.283,-0.495 a 0.262,-0.092,-0.071
9,n_activos_t0,0.000,-0.152,-0.536 a 0.233,-0.529 a 0.221,-0.152,-0.134


In [4]:
fig, ax = plt.subplots(figsize=(8.5, 5))
yy = np.arange(len(V))[::-1]
ax.errorbar(t2["coef_estandarizado"], yy + 0.18, xerr=[t2["coef_estandarizado"] - t2["ic_inf"], t2["ic_sup"] - t2["coef_estandarizado"]],
            fmt="o", color=figuras.AZUL, capsize=2, ms=4, label="M2 (IC bootstrap)")
ax.errorbar(f3.b, yy - 0.18, xerr=1.96 * de, fmt="s", color=figuras.NARANJA, capsize=2, ms=4, label="M3 posterior (IC creíble)")
ax.scatter(f3.mu, yy - 0.18, marker="x", color="black", s=28, zorder=5, label="Centro del prior (Home Credit)")
ax.axvline(0, color=figuras.GRIS, lw=1)
ax.set_yticks(yy, [f"{v} ({'+' if SIG[v] == 1 else ('−' if SIG[v] == -1 else '?')})" for v in V], fontsize=8)
ax.set_xlabel("Coeficiente estandarizado (log-odds por desviación estándar)")
ax.set_title("Qué aprendieron M2 y M3 (entre paréntesis, el signo esperado)")
ax.legend(fontsize=7.5, loc="lower right")
figuras.guardar(fig, "fase4", "M04", "coeficientes_M2_M3",
                "Ningún coeficiente excluye el 0; empleador, antigüedad, edad y monto son las señales más consistentes, y el prior desplaza la historia del cliente sin cambiar la conclusión.")

**Lectura de M3.**

- **Laplace es adecuada.** Los intervalos de Laplace y de Metropolis coinciden casi al decimal, así que la aproximación normal de la posterior funciona bien aquí.
- **Variables compartidas.** El *prior* es fuerte, pero los datos de FINTECH lo mueven en la dirección que indican por sí solos:
  - la peor mora previa baja de 0,26 a 0,11;
  - tener historia baja de 0,35 a 0,24;
  - el plazo sube de −0,09 a 0,01;
  - la antigüedad como cliente se vuelve más protectora (de −0,10 a −0,24).

  En las dos variables en que Home Credit contradice la hipótesis (`tiene_historia` y `log_plazo`), la posterior queda entre el *prior* y lo que dice M2. Es el comportamiento esperado de un *prior* con escala finita.
- **Variables propias de FINTECH** (monto, empleador, cierre previo, activos). Parten de un *prior* centrado en 0 con la misma escala τ, y sus posteriores son muy parecidas a M2: las señales del empleador y del monto sobreviven.
- **Qué significa la ventaja de M3.** Su pequeña ventaja en *log loss* viene de un efecto de edad más fuerte (−0,27 frente a −0,20) y de una mora previa positiva (0,11 frente a −0,03), los dos coherentes con el dominio. Es la clase de información que el *prior* debía aportar. Con estos datos no se puede demostrar que mejore las predicciones.

## 3. M5: SHAP del *boosting* monótono

Para el *boosting* uso SHAP (TreeExplainer), que reparte la predicción de cada crédito entre las variables. Las restricciones monótonas garantizan que el efecto de cada variable vaya en la dirección esperada; SHAP dice **cuánto** pesa cada una.

In [5]:
m5 = modelos.M5Boosting().ajustar(D, ctx)
print("Configuración elegida:", m5.elegido)
sv = shap.TreeExplainer(m5.m.m).shap_values(X)
imp = pd.DataFrame({"variable": V, "media_abs_shap": np.abs(sv).mean(axis=0),
                    "rho_valor_shap": [pd.Series(X[v]).corr(pd.Series(sv[:, j]), method="spearman") for j, v in enumerate(V)]}
                   ).sort_values("media_abs_shap", ascending=False)
io_datos.guardar_resultado(imp.round(4), "fase4", "shap_M5")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 1.4]})
axes[0].barh(imp["variable"][::-1], imp["media_abs_shap"][::-1], color=figuras.AZUL)
axes[0].set_xlabel("Media de |SHAP| (log-odds)")
axes[0].set_title("Importancia en M5")
top = imp["variable"].head(4).tolist()
for v, col in zip(top, figuras.SECUENCIA):
    j = V.index(v)
    o = np.argsort(X[v].to_numpy())
    axes[1].scatter(X[v].rank(pct=True).to_numpy()[o], sv[o, j], s=6, alpha=0.5, color=col, label=v)
axes[1].axhline(0, color=figuras.GRIS, lw=0.8)
axes[1].set_xlabel("Percentil de la variable")
axes[1].set_ylabel("SHAP (log-odds)")
axes[1].set_title("Dependencia de las cuatro variables principales")
axes[1].legend(fontsize=7.5, markerscale=2)
fig.tight_layout()
figuras.guardar(fig, "fase4", "M05", "shap_M5",
                "El boosting usa solo cinco variables (empleador, antigüedad, plazo, edad y monto), con efectos monótonos y sin interacciones.")
imp.round(4)

Configuración elegida: (1, 50, 0.05)


,variable,media_abs_shap,rho_valor_shap
10,log_emp_previos,0.2163,-0.7618
5,meses_cliente,0.1530,-0.9059
1,log_plazo,0.1132,0.4890
2,edad_t0,0.1086,-0.8457
0,log_monto,0.0472,0.8188
3,tiene_historia,0.0000,NaN
4,log_n_previos,0.0000,NaN
6,log_peor_mora_previa,0.0000,NaN
7,previo_adverso,0.0000,NaN
8,previo_favorable,0.0000,NaN


**Lectura de M5.** El pliegue interno eligió árboles de profundidad 1. El modelo es, por tanto, aditivo: una suma de escalones por variable, sin interacciones, y SHAP coincide exactamente con el efecto de cada variable.

Con 50 árboles y una tasa de 0,05, el modelo solo llega a usar **cinco variables**: la experiencia del empleador (la más importante), la antigüedad como cliente, el plazo, la edad y el monto. Las seis restantes tienen SHAP exactamente 0, porque ningún árbol las eligió para dividir.

La jerarquía coincide en lo esencial con M2: el empleador, la antigüedad y la edad protegen, y el monto suma riesgo. Que un modelo no lineal y uno lineal penalizado lleguen a las mismas señales le da robustez a la conclusión. Que el *boosting* no encuentre interacciones (profundidad 1) indica que, con 23 eventos, no hay estructura no lineal detectable que justifique un modelo más complejo.

## 4. Supervivencia: *hazard ratios* del tiempo hasta 90+

El modelo de Cox usa los 1.083 créditos de libranza 2022+ (29 eventos), incluidos los inmaduros como censurados. Reporto *hazard ratios* por desviación estándar. Es un ajuste sin penalizar: con 29 eventos y 11 variables los IC son anchos, y lo uso de forma descriptiva, como contraste de las señales de los modelos binarios.

In [6]:
S = ctx.supervivencia
XS = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit_transform(diseno.construir_X(S))
cox = sm.PHReg(S["tiempo_90"].to_numpy(), XS, status=S["evento_90"].to_numpy()).fit()
ci = cox.conf_int()
tc = pd.DataFrame({"variable": V, "hazard_ratio_por_de": np.exp(cox.params), "ic_inf": np.exp(ci[:, 0]),
                   "ic_sup": np.exp(ci[:, 1]), "p": cox.pvalues})
io_datos.guardar_resultado(tc.round(4), "fase4", "cox_hazard_ratios")

fig, ax = plt.subplots(figsize=(6.5, 4.4))
yy = np.arange(len(V))[::-1]
ax.errorbar(tc["hazard_ratio_por_de"], yy, xerr=[tc["hazard_ratio_por_de"] - tc["ic_inf"], tc["ic_sup"] - tc["hazard_ratio_por_de"]],
            fmt="o", color=figuras.VERDE, capsize=2)
ax.axvline(1, color=figuras.GRIS, lw=1)
ax.set_xscale("log")
ax.set_yticks(yy, V, fontsize=8)
ax.set_xlabel("Hazard ratio por desviación estándar (escala log)")
ax.set_title(f"Cox: tiempo hasta 90+ ({len(S)} créditos, {int(S['evento_90'].sum())} eventos)")
figuras.guardar(fig, "fase4", "M06", "cox_hazard_ratios",
                "En Cox solo dos variables son significativas: la experiencia del empleador protege (HR 0,56) y, contra la hipótesis, también el plazo largo (HR 0,44).")
tc.round(3)

,variable,hazard_ratio_por_de,ic_inf,ic_sup,p
0,log_monto,1.365,0.858,2.172,0.188
1,log_plazo,0.436,0.231,0.824,0.011
2,edad_t0,0.886,0.596,1.315,0.547
3,tiene_historia,1.277,0.477,3.414,0.626
4,log_n_previos,2.160,0.493,9.464,0.307
5,meses_cliente,0.655,0.204,2.102,0.477
6,log_peor_mora_previa,1.226,0.723,2.081,0.450
7,previo_adverso,0.855,0.400,1.826,0.686
8,previo_favorable,0.625,0.260,1.499,0.292
9,n_activos_t0,0.591,0.265,1.318,0.199


**Lectura de Cox.** Con los 1.083 créditos y 29 eventos, dos *hazard ratios* tienen IC que excluye el 1:

- **La experiencia del empleador** (HR 0,56; p = 0,004) es la señal más firme de toda la tesis. Aparece con el mismo signo en todos los modelos y aquí es significativa.
- **El plazo** (HR 0,44; p = 0,01) va en contra de la hipótesis y de los modelos binarios. Mi lectura es que, a monto constante, un plazo más largo implica una cuota más baja, y por tanto menos presión mensual sobre la nómina. Es una hipótesis que la Fase 6 debe vigilar, no un hallazgo confirmado.

El monto (HR 1,37) y la edad (HR 0,89) van en la dirección de los binarios, sin significancia.

## 5. Contraste de signos frente a la hipótesis de dominio

El signo esperado lo fijé en la Fase 1, antes de mirar la asociación con el evento. Lo contrasto con cuatro modelos. Para el *boosting* uso el signo de la correlación entre el valor de la variable y su SHAP.

In [7]:
signos = pd.DataFrame({"variable": V, "esperado": SIG[V].map({1: "+", -1: "−", 0: "?"}).to_numpy(),
                       "M2": np.sign(coef2.to_numpy()), "M3": np.sign(f3.b),
                       "M5": np.sign(imp.set_index("variable").loc[V, "rho_valor_shap"].fillna(0).to_numpy()),
                       "Cox": np.sign(cox.params)})
for c in ["M2", "M3", "M5", "Cox"]:
    signos[c] = signos[c].map({1.0: "+", -1.0: "−", 0.0: "0"})
signos["coinciden_con_esperado"] = [sum(signos.loc[i, c] == signos.loc[i, "esperado"] for c in ["M2", "M3", "M5", "Cox"])
                                    if signos.loc[i, "esperado"] != "?" else None for i in signos.index]
io_datos.guardar_resultado(signos, "fase4", "contraste_signos")
signos

,variable,esperado,M2,M3,M5,Cox,coinciden_con_esperado
0,log_monto,?,+,+,+,+,NaN
1,log_plazo,+,+,+,+,−,3.0
2,edad_t0,−,−,−,−,−,4.0
3,tiene_historia,−,+,+,0,+,0.0
4,log_n_previos,−,−,−,0,+,2.0
5,meses_cliente,−,−,−,−,−,4.0
6,log_peor_mora_previa,+,−,+,0,+,2.0
7,previo_adverso,+,+,+,0,−,2.0
8,previo_favorable,−,−,−,0,−,3.0
9,n_activos_t0,+,−,−,0,−,0.0


In [8]:
_ = io_datos.escribir_registro("11_interpretabilidad", {
    "M2_config": f"C={C:.3g}, l1_ratio={l1}", "M3_tau": m3.elegido, "M5_config": str(m5.elegido),
    "metropolis_aceptacion": round(acept / 40_000, 3), "bootstrap_coeficientes": 1000})
figuras.escribir_catalogo("fase4", "Figuras de la Fase 4")

PosixPath('resultados/fase4/catalogo_figuras.md')

## Conclusiones

- **Señales con el signo esperado en los cuatro modelos** (lineal penalizado, bayesiano, *boosting* y Cox): la **experiencia del empleador** con FINTECH, la **antigüedad como cliente** y la **edad**, las tres protectoras. El **monto** aumenta el riesgo en los cuatro.
- **La experiencia del empleador es la señal más firme:** significativa en Cox (HR 0,56) y con signo esperado en el 97 % de las réplicas de M2.
- **Historia del cliente.** Pesa poco una vez penalizada. Ni la peor mora previa interna ni los créditos activos muestran un signo estable, así que tal como están medidas no hay evidencia de que predigan el 90+.
- **El *prior* de Home Credit.** Funciona como se diseñó: aporta efectos generales (edad, mora previa) y los datos propios lo corrigen en la dirección que indican. Laplace y Metropolis coinciden.
- **Sin no linealidad.** El *boosting* elige profundidad 1, así que no hay interacciones detectables; un modelo lineal penalizado es suficiente.
- **Límite.** Todo esto son **señales**, no efectos causales ni precisos: ningún coeficiente de M2 tiene un IC que excluya el 0. El empleador y el monto están correlacionados con el calendario, y el notebook 12 los somete a sensibilidades.